# Thu thập dữ liệu CafeLand
**Phụ trách:** Hà Trọng Hữu Duy · **Chạy:** 31/08–01/09/2026 · robots.txt kiểm tra 01/09/2026

- Không cần chạy lại để tái lập kết quả: dữ liệu thô đã lưu sẵn ở `data/raw/`.
- Crawler ghi ra `cafeland.csv` trong thư mục đang chạy; nhóm đổi tên thành `data/raw/cafeland_raw.csv` trước khi làm sạch.
- Có checkpoint mỗi 100 tin và nghỉ ngẫu nhiên giữa các request để giảm tải cho máy chủ.
- Chạy lại toàn bộ mất nhiều giờ và phụ thuộc giao diện trang tại thời điểm chạy, nên số tin có thể khác.

In [1]:
!pip install -q requests beautifulsoup4 lxml pandas openpyxl tqdm

In [2]:
# ============================================================
# CAFELAND.VN CRAWLER
# ============================================================

import os
import re
import json
import time
import random
import threading
import requests
import pandas as pd

from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm.auto import tqdm


# ============================================================
# 1. CONFIG
# ============================================================

ROW = 20000

START_PAGE = 1

BASE_URL = (
    "https://nhadat.cafeland.vn/"
    "nha-dat-ban-tai-tp-ho-chi-minh/"
)

CHECKPOINT_FILE = "checkpoint_cafeland.csv"
STATE_FILE = "checkpoint_cafeland_state.json"

OUTPUT_CSV = "cafeland.csv"
OUTPUT_XLSX = "cafeland.xlsx"

COLUMNS = [
    "Tieu_de",
    "Gia_ban",
    "Dien_tich",
    "Dia_diem",
    "Loai_hinh_BDS",
    "Mo_ta_Dac_diem",
    "Nguoi_dang_Chu_dau_tu",
    "Ngay_dang",
    "URL"
]

MAX_WORKERS = 12

RETRIES = 3

TIMEOUT = 20

CHECKPOINT_EVERY = 100


# ============================================================
# 2. THREAD LOCAL SESSION
# ============================================================

_thread_local = threading.local()


def get_session():

    if not hasattr(_thread_local, "session"):

        session = requests.Session()

        retry_strategy = Retry(
            total=RETRIES,
            connect=RETRIES,
            read=RETRIES,
            status=RETRIES,
            backoff_factor=0.5,

            status_forcelist=[
                429,
                500,
                502,
                503,
                504
            ],

            allowed_methods=[
                "GET"
            ]
        )

        adapter = HTTPAdapter(
            max_retries=retry_strategy,
            pool_connections=MAX_WORKERS,
            pool_maxsize=MAX_WORKERS
        )

        session.mount(
            "https://",
            adapter
        )

        session.mount(
            "http://",
            adapter
        )

        session.headers.update({

            "User-Agent":
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/151.0.0.0 Safari/537.36",

            "Accept-Language":
                "vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7",

            "Accept":
                "text/html,application/xhtml+xml,"
                "application/xml;q=0.9,image/avif,image/webp,"
                "*/*;q=0.8"
        })

        _thread_local.session = session

    return _thread_local.session


# ============================================================
# 3. GET HTML
# ============================================================

def get_soup(url):

    try:

        response = get_session().get(
            url,
            timeout=TIMEOUT
        )

        if response.status_code != 200:
            return None

        return BeautifulSoup(
            response.content,
            "lxml"
        )

    except Exception:
        return None


# ============================================================
# 4. CLEAN TEXT
# ============================================================

def clean_text(text):

    if text is None:
        return ""

    text = str(text)

    text = text.replace(
        "\xa0",
        " "
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def normalize_url(url):

    if not url:
        return ""

    url = url.strip()

    if url.startswith("//"):
        url = "https:" + url

    elif url.startswith("/"):
        url = "https://nhadat.cafeland.vn" + url

    elif not url.startswith("http"):
        url = "https://nhadat.cafeland.vn/" + url

    return url.split("#")[0]


# ============================================================
# 5. TITLE
# ============================================================

def extract_title(soup):
    if not soup:
        return ""

    # Ưu tiên tiêu đề bài đăng trên trang chi tiết
    h1 = soup.select_one("h1.head-title")

    if h1:
        value = clean_text(
            h1.get_text(" ", strip=True)
        )

        if value:
            return value

    # Fallback: lấy title của trang
    title = soup.find("title")

    if title:
        value = clean_text(
            title.get_text(" ", strip=True)
        )

        value = re.sub(
            r"\s*\|\s*CafeLand.*$",
            "",
            value,
            flags=re.I
        )

        return value.strip()

    return ""




# ============================================================
# 6. DETAIL CONTAINER
# ============================================================

def get_detail_region(soup):

    selectors = [

        ".container.dangtin",

        ".dangtin",

        ".container .dangtin",

        ".detail-content",

        ".content-detail"
    ]

    for selector in selectors:

        node = soup.select_one(
            selector
        )

        if node:

            text = clean_text(
                node.get_text(
                    " ",
                    strip=True
                )
            )

            if len(text) > 100:

                return node

    return soup


# ============================================================
# 7. FIND VALUE BY LABEL
# ============================================================

def find_label_value(
    region,
    label,
    stop_labels=None
):

    if stop_labels is None:

        stop_labels = []

    label_lower = label.lower()

    # --------------------------------------------------------
    # Cách 1: tìm text node đúng bằng label
    # --------------------------------------------------------

    for node in region.find_all(
        string=True
    ):

        current = clean_text(
            node
        )

        if current.lower() != label_lower:
            continue

        parent = node.parent

        if not parent:
            continue

        # ----------------------------------------------------
        # Cùng parent
        # ----------------------------------------------------

        parent_text = clean_text(
            parent.get_text(
                " ",
                strip=True
            )
        )

        if len(parent_text) > len(label) + 3:

            value = re.sub(
                rf"^{re.escape(label)}\s*:?\s*",
                "",
                parent_text,
                flags=re.I
            )

            value = clean_text(
                value
            )

            if value:

                if not any(
                    stop.lower() in value.lower()
                    for stop in stop_labels
                ):

                    return value

        # ----------------------------------------------------
        # Sibling
        # ----------------------------------------------------

        for sibling in parent.find_next_siblings():

            value = clean_text(
                sibling.get_text(
                    " ",
                    strip=True
                )
            )

            if not value:
                continue

            if any(
                stop.lower() == value.lower()
                for stop in stop_labels
            ):
                break

            return value

    return ""


# ============================================================
# 8. GIÁ BÁN
# ============================================================


def extract_price(soup):
    if not soup:
        return ""

    for item in soup.select(".content .col-item, .content .col.item"):

        label = item.select_one(".infor-note")

        if not label:
            continue

        label_text = clean_text(
            label.get_text(" ", strip=True)
        ).lower()

        if label_text != "giá bán":
            continue

        value_node = item.select_one(".infor-data")

        if not value_node:
            # Một số HTML có thể dùng class "infor data"
            value_node = item.select_one(".infor.data")

        if value_node:
            value = clean_text(
                value_node.get_text(" ", strip=True)
            )

            if value and re.search(r"\d", value):
                return value

    # --------------------------------------------------------
    # Fallback: tìm label "Giá bán" trong detail region
    # --------------------------------------------------------

    region = get_detail_region(soup)

    if not region:
        return ""

    for node in region.find_all(
        string=True
    ):

        text = clean_text(node)

        if text.lower() != "giá bán":
            continue

        parent = node.parent

        if not parent:
            continue

        # Tìm giá trong cùng block
        value_node = parent.find_next(
            class_="infor-data"
        )

        if value_node:
            value = clean_text(
                value_node.get_text(" ", strip=True)
            )

            if (
                value
                and re.search(r"\d", value)
                and "m²" not in value.lower()
                and "m2" not in value.lower()
            ):
                return value

    return ""



# ============================================================
# 9. DIỆN TÍCH
# ============================================================


def extract_area(soup):
    if not soup:
        return ""

    # --------------------------------------------------------
    # Ưu tiên DOM CafeLand
    # --------------------------------------------------------

    for item in soup.select(".content .col-item, .content .col.item"):

        label = item.select_one(".infor-note")

        if not label:
            continue

        label_text = clean_text(
            label.get_text(" ", strip=True)
        ).lower()

        if label_text != "diện tích":
            continue

        # CafeLand có thể dùng:
        # .infor.data
        # hoặc .infor-data
        value_node = item.select_one(
            ".infor.data, .infor-data"
        )

        if value_node:
            value = clean_text(
                value_node.get_text(" ", strip=True)
            )

            match = re.search(
                r"([\d.,]+)\s*m(?:²|2)",
                value,
                flags=re.I
            )

            if match:
                return clean_text(
                    match.group(1) + "m2"
                )

    # --------------------------------------------------------
    # Fallback: tìm label Diện tích
    # --------------------------------------------------------

    region = get_detail_region(soup)

    if not region:
        return ""

    for node in region.find_all(string=True):

        text = clean_text(node)

        if text.lower() != "diện tích":
            continue

        parent = node.parent

        if not parent:
            continue

        value_node = parent.find_next(
            class_=lambda x: x and (
                "infor" in x and
                ("data" in x or "data" in x)
            )
        )

        if value_node:
            value = clean_text(
                value_node.get_text(" ", strip=True)
            )

            match = re.search(
                r"([\d.,]+)\s*m(?:²|2)",
                value,
                flags=re.I
            )

            if match:
                return clean_text(
                    match.group(1) + "m2"
                )

    return ""

# ============================================================
# 10. ĐỊA ĐIỂM
#
# Mục tiêu:
# chỉ trả về TP. Hồ Chí Minh
# ============================================================

def extract_location(soup):

    region = get_detail_region(
        soup
    )

    # --------------------------------------------------------
    # CafeLand hiện có:
    #
    # Vị trí:
    # Lái Thiêu
    # TP. Hồ Chí Minh
    # --------------------------------------------------------

    text = clean_text(
        region.get_text(
            " ",
            strip=True
        )
    )


    hcm_patterns = [

        "tp. hồ chí minh",
        "tp hồ chí minh",
        "tp.hồ chí minh",
        "thành phố hồ chí minh",
        "hồ chí minh",
        "tphcm",
        "tp hcm",
        "tp.hcm",
        "sài gòn",
        "saigon"
    ]

    text_lower = text.lower()

    for pattern in hcm_patterns:

        if pattern in text_lower:

            return "TP. Hồ Chí Minh"

    return ""


# ============================================================
# 11. LOẠI HÌNH BĐS

# ============================================================

def extract_property_type(
    soup,
    title,
    description,
    url
):

    title_lower = clean_text(title).lower()
    description_lower = clean_text(description).lower()
    url_lower = clean_text(url).lower()

    # --------------------------------------------------------
    # 1. DOM - Loại địa ốc
    # --------------------------------------------------------

    if soup:

        for item in soup.select(".reals-house-item"):

            label = item.select_one(".title-item")

            if not label:
                continue

            label_text = clean_text(
                label.get_text(" ", strip=True)
            ).lower()

            if label_text != "loại địa ốc":
                continue

            value_node = item.select_one(".value-item")

            if value_node:

                value = clean_text(
                    value_node.get_text(" ", strip=True)
                )

                if value:
                    value_lower = value.lower()

                    # Chuẩn hóa giá trị CafeLand
                    if "biệt thự" in value_lower or "villa" in value_lower:
                        return "Biệt thự"

                    if "shophouse" in value_lower:
                        return "Shophouse"

                    if "căn hộ" in value_lower or "chung cư" in value_lower:
                        return "Căn hộ"

                    if "nhà mặt phố" in value_lower:
                        return "Nhà mặt phố"

                    if "nhà mặt tiền" in value_lower:
                        return "Nhà mặt tiền"

                    if "nhà phố" in value_lower:
                        return "Nhà phố"

                    if "nhà riêng" in value_lower:
                        return "Nhà riêng"

                    if "đất nền" in value_lower:
                        return "Đất nền"

                    if "đất thổ cư" in value_lower:
                        return "Đất thổ cư"

                    if "nhà xưởng" in value_lower or "kho xưởng" in value_lower:
                        return "Kho - Nhà xưởng"

                    if "khách sạn" in value_lower or "nhà hàng" in value_lower:
                        return "Nhà hàng - Khách sạn"

                    return value

    # --------------------------------------------------------
    # 2. URL
    # --------------------------------------------------------

    url_patterns = [

        ("biet-thu", "Biệt thự"),
        ("villa", "Biệt thự"),

        ("shophouse", "Shophouse"),

        ("can-ho", "Căn hộ"),
        ("chung-cu", "Căn hộ"),

        ("nha-mat-pho", "Nhà mặt phố"),
        ("nha-mat-tien", "Nhà mặt tiền"),

        ("nha-pho", "Nhà phố"),
        ("nha-rieng", "Nhà riêng"),

        ("dat-nen", "Đất nền"),
        ("dat-tho-cu", "Đất thổ cư"),

        ("kho-nha-xuong", "Kho - Nhà xưởng"),
        ("nha-xuong", "Kho - Nhà xưởng"),

        ("nha-hang", "Nhà hàng - Khách sạn"),
        ("khach-san", "Nhà hàng - Khách sạn")
    ]

    for keyword, result in url_patterns:

        if keyword in url_lower:
            return result

    # --------------------------------------------------------
    # 3. TITLE
    # --------------------------------------------------------

    title_patterns = [

        ("biệt thự", "Biệt thự"),
        ("villa", "Biệt thự"),

        ("shophouse", "Shophouse"),

        ("căn hộ", "Căn hộ"),
        ("chung cư", "Căn hộ"),

        ("nhà mặt phố", "Nhà mặt phố"),
        ("nhà mặt tiền", "Nhà mặt tiền"),

        ("nhà phố", "Nhà phố"),
        ("nhà riêng", "Nhà riêng"),

        ("đất nền", "Đất nền"),
        ("đất thổ cư", "Đất thổ cư"),

        ("nhà xưởng", "Kho - Nhà xưởng"),
        ("kho xưởng", "Kho - Nhà xưởng"),

        ("khách sạn", "Nhà hàng - Khách sạn"),
        ("nhà hàng", "Nhà hàng - Khách sạn")
    ]

    for keyword, result in title_patterns:

        if keyword in title_lower:
            return result

    # --------------------------------------------------------
    # 4. DESCRIPTION
    # --------------------------------------------------------

    description_patterns = [

        ("biệt thự", "Biệt thự"),
        ("villa", "Biệt thự"),

        ("shophouse", "Shophouse"),

        ("căn hộ", "Căn hộ"),
        ("chung cư", "Căn hộ"),

        ("nhà mặt phố", "Nhà mặt phố"),
        ("nhà mặt tiền", "Nhà mặt tiền"),

        ("nhà phố", "Nhà phố"),
        ("nhà riêng", "Nhà riêng"),

        ("đất nền", "Đất nền"),
        ("đất thổ cư", "Đất thổ cư"),

        ("nhà xưởng", "Kho - Nhà xưởng"),
        ("kho xưởng", "Kho - Nhà xưởng"),

        ("khách sạn", "Nhà hàng - Khách sạn"),
        ("nhà hàng", "Nhà hàng - Khách sạn")
    ]

    for keyword, result in description_patterns:

        if keyword in description_lower:
            return result

    # --------------------------------------------------------
    # 5. Tổng quát
    # --------------------------------------------------------

    if re.search(r"\bbán nhà\b", title_lower):
        return "Nhà"

    if re.search(r"\bbán đất\b", title_lower):
        return "Đất"

    return ""


# ============================================================
# 12. MÔ TẢ
# ============================================================


def extract_description(soup):

    if not soup:
        return ""

    # --------------------------------------------------------
    # 1. DOM chính của CafeLand
    # --------------------------------------------------------

    node = soup.select_one(
        ".reals-description .blk-content.content"
    )

    if node:

        value = clean_text(
            node.get_text(
                " ",
                strip=True
            )
        )

        if value:
            return value[:12000]

    # --------------------------------------------------------
    # 2. Fallback: tìm section Thông tin mô tả
    # --------------------------------------------------------

    for section in soup.select(
        ".reals-description"
    ):

        title = section.select_one(
            ".title"
        )

        if not title:
            continue

        title_text = clean_text(
            title.get_text(
                " ",
                strip=True
            )
        ).lower()

        if title_text != "thông tin mô tả":
            continue

        content = section.select_one(
            ".blk-content"
        )

        if content:

            value = clean_text(
                content.get_text(
                    " ",
                    strip=True
                )
            )

            if value:
                return value[:12000]

    return ""



# ============================================================
# 13. NGƯỜI ĐĂNG
# ============================================================


def extract_owner(soup):

    if not soup:
        return ""

    # --------------------------------------------------------
    # DOM người đăng của CafeLand
    # --------------------------------------------------------

    owner = soup.select_one(
        ".profile-info .profile-name strong"
    )

    if owner:

        value = clean_text(
            owner.get_text(
                " ",
                strip=True
            )
        )

        if value:
            return value

    return ""

# ============================================================
# 14. NGÀY ĐĂNG
# ============================================================


def extract_post_date(soup):

    if not soup:
        return ""

    # --------------------------------------------------------
    # 1. DOM chính của CafeLand
    # --------------------------------------------------------

    date_node = soup.select_one(
        ".col-right .infor i"
    )

    if date_node:

        value = clean_text(
            date_node.get_text(
                " ",
                strip=True
            )
        )

        match = re.search(
            r"Ngày đăng\s*:?\s*"
            r"(\d{1,2}[-/]\d{1,2}[-/]\d{4})",
            value,
            flags=re.I
        )

        if match:
            return match.group(1)

    # --------------------------------------------------------
    # 2. Fallback: tìm trong detail region
    # --------------------------------------------------------

    region = get_detail_region(soup)

    if not region:
        return ""

    text = clean_text(
        region.get_text(
            " ",
            strip=True
        )
    )

    # Ngày đăng
    match = re.search(
        r"Ngày đăng\s*:?\s*"
        r"(\d{1,2}[-/]\d{1,2}[-/]\d{4})",
        text,
        flags=re.I
    )

    if match:
        return match.group(1)

    # Cập nhật
    match = re.search(
        r"Cập nhật\s*:?\s*"
        r"(\d{1,2}[-/]\d{1,2}[-/]\d{4})",
        text,
        flags=re.I
    )

    if match:
        return match.group(1)

    return ""




# ============================================================
# 15. SCRAPE DETAIL
# ============================================================

def scrape_detail(url):

    soup = get_soup(
        url
    )

    if soup is None:
        return None

    try:

        title = extract_title(
            soup
        )

        if not title:
            return None

        price = extract_price(
            soup
        )

        area = extract_area(
            soup
        )

        location = extract_location(
            soup
        )

        description = extract_description(
            soup
        )

        property_type = extract_property_type(
            soup,
            title,
            description,
            url
        )

        owner = extract_owner(
            soup
        )

        post_date = extract_post_date(
            soup
        )

        return {

            "Tieu_de":
                title,

            "Gia_ban":
                price,

            "Dien_tich":
                area,

            "Dia_diem":
                location,

            "Loai_hinh_BDS":
                property_type,

            "Mo_ta_Dac_diem":
                description,

            "Nguoi_dang_Chu_dau_tu":
                owner,

            "Ngay_dang":
                post_date,

            "URL":
                url
        }

    except Exception:

        return None


# ============================================================
# 16. GET DETAIL URLS
# ============================================================

def get_detail_urls(page):

    if page == 1:

        url = BASE_URL

    else:

        url = (
            BASE_URL.rstrip("/")
            + f"/page-{page}/"
        )

    soup = get_soup(
        url
    )

    if soup is None:

        return []

    urls = []

    seen = set()

    # --------------------------------------------------------
    # Tìm tất cả link
    # --------------------------------------------------------

    for a in soup.find_all(
        "a",
        href=True
    ):

        href = normalize_url(
            a.get("href")
        )

        if not href:
            continue

        # ----------------------------------------------------
        # Chỉ lấy link thuộc nhadat.cafeland.vn
        # ----------------------------------------------------

        if "nhadat.cafeland.vn" not in href:
            continue

        # ----------------------------------------------------
        # Loại bỏ link category/filter
        # ----------------------------------------------------

        excluded = [

            "/nha-dat-ban/",
            "/nha-dat-ban-tai-",
            "/cho-thue",
            "/du-an",
            "/blog",
            "/tim-kiem",
            "/dang-tin",
            "/dang-ky",
            "/dang-nhap"
        ]

        if any(
            x in href.lower()
            for x in excluded
        ):

            continue

        # ----------------------------------------------------
        # Tin chi tiết CafeLand thường kết thúc:
        # - số ID .html
        # ----------------------------------------------------

        if not re.search(
            r"-\d{6,}\.html$",
            href,
            flags=re.I
        ):

            continue

        if href in seen:
            continue

        seen.add(
            href
        )

        urls.append(
            href
        )

    return urls


# ============================================================
# 17. VALIDATE
# ============================================================

def validate_row(row):

    if not row:
        return False

    if not row.get(
        "URL",
        ""
    ):
        return False

    if not row.get(
        "Tieu_de",
        ""
    ):
        return False

    return True


# ============================================================
# 18. SAVE CHECKPOINT
# ============================================================

def save_checkpoint(
    rows,
    current_page
):

    if not rows:
        return

    df = pd.DataFrame(
        rows,
        columns=COLUMNS
    )

    df = df.fillna("")

    df = df.drop_duplicates(
        subset=["URL"],
        keep="first"
    )

    df.to_csv(
        CHECKPOINT_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    state = {

        "current_page":
            current_page,

        "rows":
            len(df),

        "last_saved":
            time.strftime(
                "%Y-%m-%d %H:%M:%S"
            )
    }

    with open(
        STATE_FILE,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            state,
            f,
            ensure_ascii=False,
            indent=2
        )


# ============================================================
# 19. LOAD CHECKPOINT
# ============================================================

def load_checkpoint():

    if not os.path.exists(
        CHECKPOINT_FILE
    ):

        return [], set(), START_PAGE

    try:

        df = pd.read_csv(
            CHECKPOINT_FILE,
            encoding="utf-8-sig",
            dtype=str
        )

        df = df.fillna("")

        for col in COLUMNS:

            if col not in df.columns:

                df[col] = ""

        df = df[
            COLUMNS
        ]

        df = df.drop_duplicates(
            subset=["URL"],
            keep="first"
        )

        rows = df.to_dict(
            orient="records"
        )

        urls = set(
            df["URL"].tolist()
        )

        current_page = START_PAGE

        if os.path.exists(
            STATE_FILE
        ):

            try:

                with open(
                    STATE_FILE,
                    "r",
                    encoding="utf-8"
                ) as f:

                    state = json.load(
                        f
                    )

                current_page = int(
                    state.get(
                        "current_page",
                        START_PAGE
                    )
                )

            except Exception:

                pass

        return (
            rows,
            urls,
            current_page
        )

    except Exception:

        return [], set(), START_PAGE


# ============================================================
# 20. MAIN CRAWLER
# ============================================================

def crawl_cafeland():

    rows, scraped_urls, current_page = (
        load_checkpoint()
    )

    print(
        "=" * 70
    )

    print(
        "CAFELAND.VN CRAWLER"
    )

    print(
        "=" * 70
    )

    print(
        f"Mục tiêu        : {ROW:,}"
    )

    print(
        f"Checkpoint      : {len(rows):,}"
    )

    print(
        f"Bắt đầu page    : {current_page}"
    )

    print(
        f"Workers         : {MAX_WORKERS}"
    )

    print(
        "Khu vực         : TP. Hồ Chí Minh"
    )

    print(
        "=" * 70
    )

    last_saved = len(
        rows
    )

    empty_pages = 0

    try:

        with ThreadPoolExecutor(
            max_workers=MAX_WORKERS
        ) as executor:

            while len(rows) < ROW:

                print(
                    f"\nPAGE {current_page}"
                )

                page_urls = get_detail_urls(
                    current_page
                )

                print(
                    f"URL tìm thấy: {len(page_urls)}"
                )

                if not page_urls:

                    empty_pages += 1

                    print(
                        f"Trang rỗng: {empty_pages}"
                    )

                    # tránh chạy vô hạn nếu URL phân trang
                    # không còn tồn tại

                    if empty_pages >= 3:

                        print(
                            "Không còn page có dữ liệu."
                        )

                        break

                    current_page += 1

                    continue

                empty_pages = 0

                new_urls = [

                    url

                    for url in page_urls

                    if url not in scraped_urls
                ]

                print(
                    f"URL mới: {len(new_urls)}"
                )

                if not new_urls:

                    current_page += 1

                    continue

                remaining = (
                    ROW
                    -
                    len(rows)
                )

                new_urls = new_urls[
                    :remaining
                ]

                futures = {

                    executor.submit(
                        scrape_detail,
                        url
                    ):
                    url

                    for url in new_urls
                }

                page_rows = []

                with tqdm(
                    total=len(futures),
                    desc=f"Page {current_page}",
                    unit="tin"
                ) as pbar:

                    for future in as_completed(
                        futures
                    ):

                        url = futures[
                            future
                        ]

                        try:

                            result = future.result()

                            if (
                                result
                                and
                                validate_row(
                                    result
                                )
                            ):

                                page_rows.append(
                                    result
                                )

                                scraped_urls.add(
                                    url
                                )

                        except Exception:

                            pass

                        pbar.update(
                            1
                        )

                rows.extend(
                    page_rows
                )

                # ------------------------------------------------
                # Chống trùng
                # ------------------------------------------------

                unique = {}

                for row in rows:

                    url = row.get(
                        "URL",
                        ""
                    )

                    if url:

                        unique[url] = row

                rows = list(
                    unique.values()
                )

                rows = rows[
                    :ROW
                ]

                print(
                    f"Đã lấy: {len(rows):,}/{ROW:,}"
                )

                # ------------------------------------------------
                # SAVE CHECKPOINT
                # ------------------------------------------------

                if (
                    len(rows) - last_saved
                    >= CHECKPOINT_EVERY
                    or page_rows
                ):

                    save_checkpoint(
                        rows,
                        current_page + 1
                    )

                    last_saved = len(
                        rows
                    )

                    print(
                        "✓ Đã lưu checkpoint"
                    )

                current_page += 1

                time.sleep(
                    random.uniform(
                        0.2,
                        0.5
                    )
                )

    except KeyboardInterrupt:

        print(
            "\n\nĐÃ DỪNG CRAWLER BẰNG CTRL + C"
        )

        save_checkpoint(
            rows,
            current_page
        )

        print(
            f"✓ Đã lưu {len(rows):,} dòng"
        )

        return pd.DataFrame(
            rows,
            columns=COLUMNS
        )

    except Exception as e:

        print(
            "\nCrawler lỗi:",
            repr(e)
        )

        save_checkpoint(
            rows,
            current_page
        )

        return pd.DataFrame(
            rows,
            columns=COLUMNS
        )

    # ========================================================
    # FINAL DATA
    # ========================================================

    df = pd.DataFrame(
        rows,
        columns=COLUMNS
    )

    df = df.fillna("")

    df = df.drop_duplicates(
        subset=["URL"],
        keep="first"
    )

    df = df.head(
        ROW
    )

    df = df.reset_index(
        drop=True
    )

    # ========================================================
    # SAVE CSV
    # ========================================================

    df.to_csv(
        OUTPUT_CSV,
        index=False,
        encoding="utf-8-sig"
    )

    # ========================================================
    # SAVE EXCEL
    # ========================================================

    df.to_excel(
        OUTPUT_XLSX,
        index=False
    )

    # ========================================================
    # FINAL CHECKPOINT
    # ========================================================

    save_checkpoint(
        df.to_dict(
            orient="records"
        ),
        current_page
    )

    # ========================================================
    # REPORT
    # ========================================================

    print(
        "\n"
        + "=" * 70
    )

    print(
        "CRAWL HOÀN TẤT"
    )

    print(
        "=" * 70
    )

    print(
        f"Số dòng: {len(df):,}"
    )

    print(
        f"Số cột: {len(df.columns)}"
    )

    print(
        f"CSV: {OUTPUT_CSV}"
    )

    print(
        f"Excel: {OUTPUT_XLSX}"
    )

    print(
        f"Checkpoint: {CHECKPOINT_FILE}"
    )

    print(
        "\nSố lượng thiếu:"
    )

    print(
        df.isna().sum()
    )

    print(
        "\n10 dòng đầu:"
    )

    display(
        df.head(10)
    )

    return df


# ============================================================
# 21. START
# ============================================================

df = crawl_cafeland()

CAFELAND.VN CRAWLER
Mục tiêu        : 20,000
Checkpoint      : 0
Bắt đầu page    : 1
Workers         : 12
Khu vực         : TP. Hồ Chí Minh

PAGE 1
URL tìm thấy: 40
URL mới: 40


Page 1:   0%|          | 0/40 [00:00<?, ?tin/s]

Đã lấy: 39/20,000
✓ Đã lưu checkpoint

PAGE 2
URL tìm thấy: 30
URL mới: 26


Page 2:   0%|          | 0/26 [00:00<?, ?tin/s]

Đã lấy: 63/20,000
✓ Đã lưu checkpoint

PAGE 3
URL tìm thấy: 32
URL mới: 28


...

Đã lấy: 278/20,000
✓ Đã lưu checkpoint

PAGE 14
URL tìm thấy: 37
URL mới: 30


Page 14:   0%|          | 0/30 [00:00<?, ?tin/s]

Đã lấy: 305/20,000
✓ Đã lưu checkpoint

PAGE 15
URL tìm thấy: 40
URL mới: 31


Page 15:   0%|          | 0/31 [00:00<?, ?tin/s]



ĐÃ DỪNG CRAWLER BẰNG CTRL + C
✓ Đã lưu 305 dòng
